# GLTC-TNN ????????

?? `data` ?????????????? `results/GLTC-TNN`????? MSE?RMSE?RSE?MAE ??????

In [1]:

import numpy as np
import pandas as pd
import time
from pathlib import Path


def ten2mat(tensor, mode):
    return np.reshape(np.moveaxis(tensor, mode, 0), (tensor.shape[mode], -1), order='F')


def mat2ten(mat, tensor_size, mode):
    index = [mode]
    for i in range(tensor_size.shape[0]):
        if i != mode:
            index.append(i)
    return np.moveaxis(np.reshape(mat, list(tensor_size[index]), order='F'), 0, mode)


def svt_tnn(mat, tau, theta):
    u, s, v = np.linalg.svd(mat, full_matrices=False)
    vec = np.zeros(len(s))
    for i in range(len(s)):
        if i >= theta:
            vec[i] = s[i] - tau
        else:
            vec[i] = s[i]
    vec[vec < 0] = 0
    return np.matmul(np.matmul(u, np.diag(vec)), v)


In [2]:

def GLTC_TNN(dense_tensor, sparse_tensor, Omega, alpha, beta, rho, theta, maxiter, verbose=True):
    Omega = Omega.astype(bool)
    dim0 = sparse_tensor.ndim
    dim1, dim2, dim3 = sparse_tensor.shape
    dim = np.array([dim1, dim2, dim3])
    binary_tensor = Omega.astype(float)
    sparse_tensor = sparse_tensor.astype(float)
    tensor_hat = sparse_tensor.copy()
    X = np.zeros((dim1, dim2, dim3, dim0))
    Z = np.zeros((dim1, dim2, dim3, dim0))
    T = np.zeros((dim1, dim2, dim3, dim0))
    for k in range(dim0):
        X[:, :, :, k] = tensor_hat
        Z[:, :, :, k] = tensor_hat

    D1 = np.zeros((dim1 - 1, dim1))
    for i in range(dim1 - 1):
        D1[i, i] = -1
        D1[i, i + 1] = 1
    D2 = np.zeros((dim2 - 1, dim2))
    for i in range(dim2 - 1):
        D2[i, i] = -1
        D2[i, i + 1] = 1

    mse_list, rmse_list, rse_list, mae_list, rel_list, elapsed_list = [], [], [], [], [], []
    start_time = time.perf_counter()
    last_tensor = tensor_hat.copy()
    dense_norm = max(np.linalg.norm(dense_tensor.ravel()), np.finfo(float).eps)

    for iters in range(maxiter):
        for k in range(dim0):
            Z_hat = ten2mat(X[:, :, :, k] + T[:, :, :, k] / rho, k)
            Z_hat = svt_tnn(Z_hat, alpha / rho, theta)
            Z[:, :, :, k] = mat2ten(Z_hat, dim, k)
            var = ten2mat(rho * Z[:, :, :, k] - T[:, :, :, k], k)
            if k == 0:
                var0 = mat2ten(np.linalg.solve(beta * D1.T @ D1 + rho * np.eye(dim1), var), dim, k)
            elif k == 1:
                var0 = mat2ten(np.linalg.solve(beta * D2.T @ D2 + rho * np.eye(dim2), var), dim, k)
            else:
                var0 = Z[:, :, :, k] - T[:, :, :, k] / rho
            X[:, :, :, k] = np.multiply(1 - binary_tensor, var0) + sparse_tensor

        tensor_hat = np.mean(X, axis=3)
        tensor_hat = np.nan_to_num(tensor_hat, nan=0.0, posinf=1.0, neginf=0.0)
        tensor_hat = np.clip(tensor_hat, 0, 1)
        diff = dense_tensor.astype(float) - tensor_hat.astype(float)
        mse_value = float(np.mean(diff ** 2))
        rmse_value = float(np.sqrt(mse_value))
        rse_value = float(np.linalg.norm(diff.ravel()) / dense_norm)
        mae_value = float(np.mean(np.abs(diff)))
        rel_value = float(np.linalg.norm(tensor_hat.ravel() - last_tensor.ravel()) / max(np.linalg.norm(last_tensor.ravel()), np.finfo(float).eps))
        last_tensor = tensor_hat.copy()
        elapsed_list.append(time.perf_counter() - start_time)
        mse_list.append(mse_value)
        rmse_list.append(rmse_value)
        rse_list.append(rse_value)
        mae_list.append(mae_value)
        rel_list.append(rel_value)
        if verbose:
            print(f'Epoch = {iters + 1}, MSE = {mse_value:.10f}, RMSE = {rmse_value:.10f}')
        for k in range(dim0):
            T[:, :, :, k] = T[:, :, :, k] + rho * (X[:, :, :, k] - Z[:, :, :, k])
            X[:, :, :, k] = tensor_hat.copy()

    history = pd.DataFrame({
        'iteration': np.arange(1, len(mse_list) + 1),
        'elapsed_time_seconds': np.asarray(elapsed_list, dtype=float),
        'MSE': np.asarray(mse_list, dtype=float),
        'RMSE': np.asarray(rmse_list, dtype=float),
        'RSE': np.asarray(rse_list, dtype=float),
        'MAE': np.asarray(mae_list, dtype=float),
        'relative_change': np.asarray(rel_list, dtype=float),
    })
    return tensor_hat, history


In [3]:

base_dir = Path.cwd()
data_dir = base_dir / 'data'
result_root = base_dir / 'results' / 'GLTC-TNN'
result_root.mkdir(parents=True, exist_ok=True)

seed_list = [920, 921, 922]
miss_list = [0.8, 0.9, 0.95]
alpha = 10
rho = 1.001
beta = 0.1 * rho
theta = 100
maxiter = 300
all_summary = []


def enrich_history(history, data_name, seed, missing_rate):
    history = history.copy()
    history.insert(0, 'dataset', data_name)
    history.insert(1, 'method', 'GLTC-TNN')
    history.insert(2, 'variant', 'default')
    history.insert(3, 'seed', seed)
    history.insert(4, 'missing_rate', missing_rate)
    history['parameter_settings'] = f'alpha={alpha}, beta={beta}, rho={rho}, theta={theta}, maxiter={maxiter}'
    history['convergence_status'] = 'ok'
    return history


for seed in seed_list:
    for missing_rate in miss_list:
        miss_tag = int(round(missing_rate * 100))
        data_name = f'S{seed}_miss{miss_tag}'
        data = np.load(data_dir / f'{data_name}.npz', allow_pickle=True)
        Xtrue = data['X'].astype(float)
        Omega = data['Omega'].astype(bool)
        Y = data['Y'].astype(float)
        result_dir = result_root / data_name
        result_dir.mkdir(parents=True, exist_ok=True)

        print('\n' + '=' * 60)
        print('Method: GLTC-TNN')
        print(f'Data: {data_name}')
        print(f'Size: {Xtrue.shape}')
        print(f'Missing rate: {missing_rate:.2f}')
        print(f'Observed entries: {int(Omega.sum())} / {Omega.size}')
        print('=' * 60)

        start = time.perf_counter()
        Xhat, history = GLTC_TNN(Xtrue, Y, Omega, alpha, beta, rho, theta, maxiter, verbose=False)
        elapsed = time.perf_counter() - start
        Xhat = np.clip(Xhat, 0, 1)
        diff = Xtrue - Xhat
        mse = float(np.mean(diff ** 2))
        rmse = float(np.sqrt(mse))
        rse = float(np.linalg.norm(diff.ravel()) / max(np.linalg.norm(Xtrue.ravel()), np.finfo(float).eps))
        mae = float(np.mean(np.abs(diff)))
        rel = float(history['relative_change'].iloc[-1]) if len(history) else np.nan
        history = enrich_history(history, data_name, seed, missing_rate)
        history.to_csv(result_dir / '\u6700\u4f73\u8fed\u4ee3.csv', index=False, encoding='utf-8-sig')
        history.to_csv(result_dir / 'best_iteration_history.csv', index=False, encoding='utf-8-sig')

        summary = pd.DataFrame([{
            'Data': data_name,
            'Method': 'GLTC-TNN',
            'Variant': 'default',
            'MissingRate': missing_rate,
            'Seed': seed,
            'alpha': alpha,
            'beta': beta,
            'rho': rho,
            'theta': theta,
            'maxiter': maxiter,
            'MSE': mse,
            'RMSE': rmse,
            'RSE': rse,
            'MAE': mae,
            'RelativeChange': rel,
            'Time': elapsed,
        }])
        summary.to_csv(result_dir / 'summary.csv', index=False, encoding='utf-8-sig')
        summary.to_excel(result_dir / '\u5b9e\u9a8c\u603b\u7ed3.xlsx', index=False)
        np.savez_compressed(result_dir / 'result.npz', data_name=data_name, X=Xtrue, Omega=Omega, Y=Y, recovered=Xhat, alpha=alpha, beta=beta, rho=rho, theta=theta, maxiter=maxiter, missing_rate=missing_rate, seed=seed)
        with open(result_dir / 'metrics.txt', 'w', encoding='utf-8') as f:
            f.write(f'data={data_name}\nmethod=GLTC-TNN\nvariant=default\nseed={seed}\nmissing_rate={missing_rate:.2f}\n')
            f.write(f'alpha={alpha}\nbeta={beta}\nrho={rho}\ntheta={theta}\nmaxiter={maxiter}\n')
            f.write(f'MSE={mse:.10f}\nRMSE={rmse:.10f}\nRSE={rse:.10f}\nMAE={mae:.10f}\nrelative_change={rel:.10f}\ntime={elapsed:.6f}\nstatus=ok\n')
        all_summary.append(summary.iloc[0].to_dict())
        print(f'MSE={mse:.10f}, RMSE={rmse:.10f}, RSE={rse:.10f}, Time={elapsed:.2f}s')

all_table = pd.DataFrame(all_summary)
all_table.to_csv(result_root / 'all_summary.csv', index=False, encoding='utf-8-sig')
all_table.to_excel(result_root / '\u5168\u90e8\u5b9e\u9a8c\u603b\u7ed3.xlsx', index=False)
print('All GLTC-TNN low rank tensor experiments finished.')



Method: GLTC-TNN
Data: S920_miss80
Size: (200, 200, 30)
Missing rate: 0.80
Observed entries: 239385 / 1200000


MSE=0.0056372468, RMSE=0.0750816006, RSE=0.1381559967, Time=117.43s

Method: GLTC-TNN
Data: S920_miss90
Size: (200, 200, 30)
Missing rate: 0.90
Observed entries: 120146 / 1200000


MSE=0.0092166964, RMSE=0.0960036271, RSE=0.1766541560, Time=116.92s

Method: GLTC-TNN
Data: S920_miss95
Size: (200, 200, 30)
Missing rate: 0.95
Observed entries: 60124 / 1200000


MSE=0.0212229512, RMSE=0.1456809912, RSE=0.2680643777, Time=117.43s

Method: GLTC-TNN
Data: S921_miss80
Size: (200, 200, 30)
Missing rate: 0.80
Observed entries: 239635 / 1200000


MSE=0.0048401901, RMSE=0.0695714746, RSE=0.1409968704, Time=117.97s

Method: GLTC-TNN
Data: S921_miss90
Size: (200, 200, 30)
Missing rate: 0.90
Observed entries: 119616 / 1200000


MSE=0.0078023324, RMSE=0.0883308126, RSE=0.1790154399, Time=116.79s

Method: GLTC-TNN
Data: S921_miss95
Size: (200, 200, 30)
Missing rate: 0.95
Observed entries: 60004 / 1200000


MSE=0.0178298741, RMSE=0.1335285515, RSE=0.2706153343, Time=118.41s

Method: GLTC-TNN
Data: S922_miss80
Size: (200, 200, 30)
Missing rate: 0.80
Observed entries: 239663 / 1200000


MSE=0.0028489079, RMSE=0.0533751616, RSE=0.0901499047, Time=117.70s

Method: GLTC-TNN
Data: S922_miss90
Size: (200, 200, 30)
Missing rate: 0.90
Observed entries: 119865 / 1200000


MSE=0.0054851895, RMSE=0.0740620650, RSE=0.1250897965, Time=117.98s

Method: GLTC-TNN
Data: S922_miss95
Size: (200, 200, 30)
Missing rate: 0.95
Observed entries: 60150 / 1200000


MSE=0.0198518212, RMSE=0.1408964911, RSE=0.2379722114, Time=117.28s
All GLTC-TNN low rank tensor experiments finished.
